# Inference Monitoring Notebook

> 📊 **Best viewed on [nbviewer](https://nbviewer.org/github/aws-samples/sample-mlops-bestpractices/blob/main/sagemaker-automated-drift-and-trend-monitoring/notebooks/3_inference_monitoring.ipynb)** — GitHub's renderer strips the JavaScript that powers Evidently's interactive drift reports. nbviewer (run by Project Jupyter) renders them in full. Use GitHub for code review; nbviewer for results.

Post-deployment monitoring for the fraud detection endpoint. Linear flow:

1. **Setup** — load config, connect AWS clients
2. **Generate drifted data** — fabricate test data that exercises drift detection
3. **Send predictions** — invoke the endpoint with sample + drifted data, all in one block
4. **Apply ground truth** — simulate fraud-investigation labels and MERGE into Athena
5. **Monitoring infrastructure** — Lake Formation grants + Lambda deploys (idempotent)
6. **Compute drift** — data drift vs. frozen `training_data` baseline + model drift vs. frozen `evaluation_data` baseline
7. **Schedule automation** — EventBridge daily, SNS alerts on threshold breach

> **Rule**: never send predictions to the endpoint *after* running the simulator (Section 4) — those rows will have NULL ground_truth and skew model-drift metrics. The cell order below enforces this: all inference invocations happen in Section 3, before any labeling.


## 1. Setup

In [ ]:
# Install project + drift-monitoring dependencies into the kernel.
# Idempotent — fast on re-runs since pip detects everything's already installed.
# Required on first run of a fresh JupyterLab kernel (SageMaker base image
# doesn't ship with evidently / awswrangler / pyathena / etc).
!pip install -e ../. --quiet


In [1]:
import sys, os, json, time, logging, uuid
from pathlib import Path
from datetime import datetime, timedelta

import boto3
import pandas as pd
import numpy as np

# Find project root and add to path
project_root = Path.cwd()
while not (project_root / '.env').exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

from src.config.config import (
    ATHENA_DATABASE, DATA_S3_BUCKET, AWS_DEFAULT_REGION,
    ATHENA_OUTPUT_S3,
    ATHENA_TRAINING_TABLE, ATHENA_EVALUATION_TABLE,
    ATHENA_INFERENCE_TABLE,
    ATHENA_GROUND_TRUTH_UPDATES_TABLE,
    CSV_DRIFTED_DATA,
    TRAINING_FEATURES, TARGET_COLUMN,
    SAGEMAKER_EXEC_ROLE, MLFLOW_MODEL_NAME,
    MONITORING_DATA_DRIFT_LOOKBACK_DAYS,
    GROUND_TRUTH_SIM_ACCURACY,
    GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT,
    GROUND_TRUTH_SIM_MODEL_DRIFT_MAG,
)
from src.train_pipeline.athena.athena_client import AthenaClient

# Data source policy:
#   - training_data, evaluation_data, monitoring_responses, inference_responses
#     are read from Athena (single source of truth across runs).
#   - drifted samples (cell 11) are read from the local CSV that cells 6/8
#     just wrote — they are a transient per-run artifact, not durable state.

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

ENDPOINT_NAME = 'fraud-detector-endpoint'
REGION = AWS_DEFAULT_REGION

runtime_client = boto3.client('sagemaker-runtime', region_name=REGION)
sm_client = boto3.client('sagemaker', region_name=REGION)
athena_client = AthenaClient()

print(f'Region: {REGION}')
print(f'Endpoint: {ENDPOINT_NAME}')
print(f'Athena DB: {ATHENA_DATABASE}')
print(f'S3 Bucket: {DATA_S3_BUCKET}')
print(f'MLflow URI: {os.getenv("MLFLOW_TRACKING_URI", "NOT SET")}')
print(f'Data Drift Lookback: {MONITORING_DATA_DRIFT_LOOKBACK_DAYS} days')


2026-06-26 21:04:52,411 - INFO - Initialized AthenaClient for database: fraud_detection


Region: us-west-2
Endpoint: fraud-detector-endpoint
Athena DB: fraud_detection
S3 Bucket: fraud-detection-monitoring-data-<ACCOUNT_ID>
MLflow URI: arn:aws:sagemaker:us-west-2:<ACCOUNT_ID>:mlflow-tracking-server/fraud-detection-monitoring-mlflow
Data Drift Lookback: 7 days


In [2]:
# Configuration Constants (loaded from .env)
# These variables can be overridden in .env file

# Lambda Functions
DRIFT_LAMBDA_NAME = os.getenv('DRIFT_LAMBDA_NAME', 'fraud-detection-drift-monitor')
MONITORING_WRITER_LAMBDA = os.getenv('MONITORING_WRITER_LAMBDA_NAME', 'fraud-monitoring-results-writer')

# SQS Queues
MONITORING_SQS_QUEUE_NAME = os.getenv('MONITORING_SQS_QUEUE_NAME', 'fraud-monitoring-results')

# SNS Topics
SNS_TOPIC_NAME = os.getenv('SNS_TOPIC_NAME', 'fraud-detection-drift-alerts')

# EventBridge Rules
EVENTBRIDGE_RULE_NAME = os.getenv('EVENTBRIDGE_RULE_NAME', 'fraud-detection-drift-check')

# CloudWatch
CLOUDWATCH_DASHBOARD_NAME = os.getenv('CLOUDWATCH_DASHBOARD_NAME', 'FraudDetection-DriftMonitoring')
CLOUDWATCH_NAMESPACE = os.getenv('CLOUDWATCH_NAMESPACE', 'FraudDetection/DriftMonitoring')
CLOUDWATCH_LOG_GROUP_DRIFT = os.getenv('CLOUDWATCH_LOG_GROUP_DRIFT', f'/aws/lambda/{DRIFT_LAMBDA_NAME}')

# Athena Tables
MONITORING_TABLE_NAME = os.getenv('MONITORING_TABLE_NAME', 'monitoring_responses')

print('Configuration loaded:')
print(f'  Drift Lambda: {DRIFT_LAMBDA_NAME}')
print(f'  Monitoring Writer: {MONITORING_WRITER_LAMBDA}')
print(f'  Monitoring Table: {ATHENA_DATABASE}.{MONITORING_TABLE_NAME}')
print(f'  SNS Topic: {SNS_TOPIC_NAME}')
print(f'  EventBridge Rule: {EVENTBRIDGE_RULE_NAME}')

# Export to the process env so library code that reads these directly
# (e.g. src.drift_monitoring.lambda_drift_monitor.load_baseline_from_registry)
# resolves the right endpoint + model package group.
os.environ.setdefault('ENDPOINT_NAME', ENDPOINT_NAME)
os.environ.setdefault('MODEL_PACKAGE_GROUP', 'fraud-detection')


Configuration loaded:
  Drift Lambda: fraud-detection-drift-monitor
  Monitoring Writer: fraud-monitoring-results-writer
  Monitoring Table: fraud_detection.monitoring_responses
  SNS Topic: fraud-detection-drift-alerts
  EventBridge Rule: fraud-detection-drift-check


'fraud-detection'

## 2. Generate Drifted Test Data

Fabricate a drifted CSV the drift detector should flag. Skip this section if you only want to test the no-drift happy path. Drift parameters live in `src/config/config.yaml` → `drift_generation.default_drift`.


In [3]:
# Generate drifted dataset (configured via config.yaml)
import subprocess
import sys

print('=' * 80)
print('GENERATING DRIFTED TEST DATA')
print('=' * 80)
print('\nThis will create data/creditcard_drifted.csv with configurable drift amounts.')
print('Drift parameters are read from: src/config/config.yaml → drift_generation.default_drift\n')

result = subprocess.run(
    [sys.executable, 'src/drift_monitoring/generate_drift_dataset.py'],
    cwd=Path.cwd().parent,  # Run from project root
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print('❌ Error generating drifted dataset:')
    print(result.stderr)
else:
    print('\n✅ Drifted dataset generated successfully!')
    print('   File: data/creditcard_drifted.csv')
    print('   To use this data, send it to your endpoint for inference testing.')

GENERATING DRIFTED TEST DATA

This will create data/creditcard_drifted.csv with configurable drift amounts.
Drift parameters are read from: src/config/config.yaml → drift_generation.default_drift



  Local training data not found, using S3 source: s3://fraud-detection-monitoring-data-<ACCOUNT_ID>/fraud-detection/data/creditcard_predictions_final.csv
GENERATING DRIFTED DATASET

Loading original dataset from: s3://fraud-detection-monitoring-data-<ACCOUNT_ID>/fraud-detection/data/creditcard_predictions_final.csv
Original dataset shape: (284807, 35)

Sampling 5000 random rows...
Sampled dataset shape: (5000, 35)

Applying feature drift:
--------------------------------------------------------------------------------
  transaction_amount:
    Original mean: 91.5086
    Drifted mean: 100.6719
    Change: +10.01%
    Description: Increased transaction amounts (inflation/behavior change)
  transaction_timestamp:
    Original mean: 95762.3350
    Drifted mean: 105760.3329
    Change: +10.44%
    Description: Time shift to simulate future period
  distance_from_home_km:
    Original mean: -0.0080
    Drifted mean: 0.5302
    Change: -6768.71%
    Description: Increased distance from home (

## 3. Send Predictions to the Endpoint

Every cell in this section invokes the deployed endpoint. The custom handler
auto-logs each prediction to Athena via SQS → Lambda → `inference_responses`
(batches of 10 records or every 30 seconds).

To make drift detection in Section 6 meaningful, **most invocations send
intentionally-drifted samples** (from Section 2). A small training-data
sanity check stays in 3.2 so you can confirm the endpoint is alive.


### 3.1 Helper: JSON invocation function

Defines `invoke_endpoint_json(endpoint_name, feature_dict)` — used by every
subsequent inference cell. Returns `(result, latency_ms)`.


In [4]:
def invoke_endpoint_json(endpoint_name, feature_dict):
    """Invoke endpoint with JSON format (custom handler with Athena logging)."""
    import json
    import time
    
    payload = json.dumps(feature_dict)
    start = time.time()
    
    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',  # Custom handler expects JSON
        Body=payload
    )
    
    latency_ms = (time.time() - start) * 1000
    result = json.loads(response['Body'].read().decode())
    
    # Custom handler returns: {"predictions": [0], "probabilities": {"fraud": [0.1234], "non_fraud": [0.8766]}}
    fraud_prob = result["probabilities"]["fraud"][0]
    prediction = result["predictions"][0]
    
    return {'prediction': prediction, 'fraud_probability': fraud_prob}, latency_ms


### 3.2 Sanity check: invoke the endpoint with one hard-coded payload

A single non-drifted prediction to confirm the endpoint container is alive
and the JSON contract works. Does NOT load any datasets — sends one literal
dict. The prediction lands in `inference_responses` like any other.


In [5]:
# Single non-drifted prediction — uses a hardcoded payload, not Athena data.
import json

sanity_payload = {
    "transaction_hour": 14, "transaction_day_of_week": 2,
    "transaction_amount": 149.62, "transaction_type_code": 1,
    "customer_age": 42, "customer_gender": 0,
    "customer_tenure_months": 36, "account_age_days": 1095,
    "distance_from_home_km": 5.2, "distance_from_last_transaction_km": 2.3,
    "time_since_last_transaction_min": 120,
    "online_transaction": 1, "international_transaction": 0,
    "high_risk_country": 0, "merchant_category_code": 5411,
    "merchant_reputation_score": 0.85, "chip_transaction": 1,
    "pin_used": 1, "card_present": 1, "cvv_match": 1,
    "address_verification_match": 1,
    "num_transactions_24h": 3, "num_transactions_7days": 12,
    "avg_transaction_amount_30days": 125.50,
    "max_transaction_amount_30days": 450.00,
    "velocity_score": 0.3, "recurring_transaction": 0,
    "previous_fraud_incidents": 0,
    "credit_limit": 5000.0, "available_credit_ratio": 0.75,
}

result, latency_ms = invoke_endpoint_json(ENDPOINT_NAME, sanity_payload)
print(f"Prediction       : {result['prediction']}")
print(f"Fraud probability: {result['fraud_probability']:.4f}")
print(f"Latency          : {latency_ms:.0f} ms")
print(f"\nLogged to {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE} (visible after SQS flush).")


Prediction       : 0
Fraud probability: 0.1931
Latency          : 8769 ms

Logged to fraud_detection.inference_responses (visible after SQS flush).


### 3.3 Bulk drifted inferences (latency + drift signal)

Loads the drifted CSV produced in Section 2, samples `NUM_BULK_TESTS` rows,
and sends them to the endpoint. Reports latency percentiles and writes the
predictions to `inference_responses` so the drift detector in Section 6 can
see them.

If Section 2 was skipped, this cell prints a warning and exits without sending
anything — go run Section 2, then come back.


In [6]:
# Bulk drifted inferences — requires Section 2 to have generated the drifted CSV.
NUM_BULK_TESTS = 101

if not CSV_DRIFTED_DATA.exists():
    print(f'⚠ Drifted CSV not found: {CSV_DRIFTED_DATA}')
    print('  Run Section 2 first to generate it, then re-run this cell.')
else:
    drifted_df = pd.read_csv(CSV_DRIFTED_DATA)
    for col in TRAINING_FEATURES:
        if col in drifted_df.columns:
            drifted_df[col] = pd.to_numeric(drifted_df[col], errors='coerce')
    drifted_df = drifted_df.dropna(subset=TRAINING_FEATURES, how='all')

    n = min(NUM_BULK_TESTS, len(drifted_df))
    samples = drifted_df.sample(n=n, random_state=123)

    print(f'Source           : {CSV_DRIFTED_DATA.name}')
    print(f'Sending          : {n} rows to {ENDPOINT_NAME}')
    print()

    predictions, latencies, errors = [], [], []
    for _, row in samples.iterrows():
        try:
            result, lat = invoke_endpoint_json(ENDPOINT_NAME, row[TRAINING_FEATURES].to_dict())
            predictions.append(result)
            latencies.append(lat)
        except Exception as e:
            errors.append(str(e))

    lat_arr = np.array(latencies) if latencies else np.array([0])
    fraud = sum(1 for p in predictions if p['prediction'] == 1)
    print(f'Sent             : {len(predictions)}/{n}  ({len(errors)} errors)')
    print(f'Latency P50/P95/P99: {np.percentile(lat_arr, 50):.0f} / {np.percentile(lat_arr, 95):.0f} / {np.percentile(lat_arr, 99):.0f} ms')
    print(f'Fraud predictions: {fraud}/{len(predictions)} ({fraud / max(len(predictions), 1) * 100:.1f}%)')


Source           : creditcard_drifted.csv
Sending          : 101 rows to fraud-detector-endpoint



Sent             : 101/101  (0 errors)
Latency P50/P95/P99: 52 / 96 / 155 ms
Fraud predictions: 0/101 (0.0%)


### 3.4 Verify predictions landed in Athena

Counts rows in `inference_responses` to confirm the SQS → Lambda writer
flushed everything we sent. If the count looks low, wait ~30 seconds and
re-run (SQS batches up to 30s before the writer Lambda flushes).


In [7]:
# Verify predictions landed in Athena.
query = f"""
SELECT COUNT(*) AS total,
       CAST(MIN(request_timestamp) AS TIMESTAMP(3)) AS earliest,
       CAST(MAX(request_timestamp) AS TIMESTAMP(3)) AS latest
FROM {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
WHERE endpoint_name = '{ENDPOINT_NAME}'
"""
result = athena_client.execute_query(query)
if not result.empty and result['total'].iloc[0] > 0:
    print(f"✓ Found {result['total'].iloc[0]:,} predictions for {ENDPOINT_NAME}")
    print(f"  Time range: {result['earliest'].iloc[0]} → {result['latest'].iloc[0]}")
else:
    print('⚠ No predictions found yet — wait ~30s for the SQS→Lambda flush, then retry.')


2026-06-26 21:05:16,673 - INFO - Executing query: 
SELECT COUNT(*) AS total,
       CAST(MIN(request_timestamp) AS TIMESTAMP(3)) AS earliest,
       C...


2026-06-26 21:05:19,206 - INFO - Initializing a Ray instance


2026-06-26 21:05:21,411	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 891269120 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=2.17gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-06-26 21:05:22,561	INFO worker.py:2007 -- Started a local Ray instance.


/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2026-06-26 21:05:23,957 - INFO - Query returned 1 rows


✓ Found 113 predictions for fraud-detector-endpoint
  Time range: 2026-06-26 20:37:16.460000 → 2026-06-26 21:05:12.147000


## 4. Apply Ground Truth

In production, ground truth arrives asynchronously from fraud investigations. For dev/test we simulate it here.

**Do not re-run Section 3 cells after this point** — newly logged predictions will have NULL ground_truth and won't contribute to model-drift metrics until the simulator runs again.


### 4.1 Simulator parameters

`accuracy`, `feature_drift_*`, and `model_drift_*` come from
`src/config/config.yaml` → `ground_truth_simulation`. The simulator
uses these to fabricate plausible labels (true/false positives /
negatives) so we can compute model-drift metrics offline.


In [8]:
# Ground truth simulation parameters from src/config/config.yaml → ground_truth_simulation.
# Only two knobs reduce simulated accuracy:
#   effective_accuracy = max(0.5, base_accuracy
#                                 - GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT
#                                 - GROUND_TRUTH_SIM_MODEL_DRIFT_MAG)
print(f'Base accuracy        : {GROUND_TRUTH_SIM_ACCURACY:.2f}')
print(f'Feature-drift impact : -{GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT:.2f}')
print(f'Model-drift magnitude: -{GROUND_TRUTH_SIM_MODEL_DRIFT_MAG:.2f}')
effective = max(0.5, GROUND_TRUTH_SIM_ACCURACY - GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT - GROUND_TRUTH_SIM_MODEL_DRIFT_MAG)
print(f'\nEffective accuracy   : {effective:.2f}  → ~{int((1 - effective) * 100)}% of labels will be flipped')


Base accuracy        : 0.85
Feature-drift impact : -0.00
Model-drift magnitude: -0.00

Effective accuracy   : 0.85  → ~15% of labels will be flipped


### 4.2 Generate and apply simulated labels

Two-step process: (1) the simulator inserts rows into
`ground_truth_updates` keyed on `inference_id`; (2) the updater issues
an Iceberg `MERGE` that writes `ground_truth` back to
`inference_responses` for each row that doesn't already have one.

**Do NOT re-run Section 3 after this point** — newly-logged predictions
will have `ground_truth = NULL` and won't contribute to model drift
until you run the simulator again.


In [9]:
# SQS → Lambda inference logging is async (CFN-deployed InferenceLoggerFunction
# batches messages: flushes every 10 records OR every 30 seconds, whichever first).
# After Section 3.3's bulk test, wait one full flush cycle so the LAST record's
# time-based flush has fired — otherwise the simulator's WHERE ground_truth IS NULL
# query will miss those rows and they'll stay unlabeled until the next sim run.
import time
print('Waiting 60s for the final SQS→Lambda flush cycle...')
time.sleep(60)

# Option 2: Simulate ground truth programmatically (in notebook)
from src.drift_monitoring.simulate_ground_truth_from_athena import GroundTruthSimulator

# Create simulator with configured drift parameters
simulator = GroundTruthSimulator(
    athena_client=athena_client,
    endpoint_name=ENDPOINT_NAME,
    accuracy=GROUND_TRUTH_SIM_ACCURACY,
    fraud_confirmation_days=(1, 7),
    non_fraud_confirmation_days=(1, 30),
    feature_drift_impact=GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT,
    model_drift_magnitude=GROUND_TRUTH_SIM_MODEL_DRIFT_MAG,
    seed=42
)

# Run simulation
print("Simulating ground truth for predictions without ground truth...")
stats = simulator.simulate_and_write(limit=None)  # Set limit=100 to only process 100 predictions

print("\n" + "=" * 80)
print("Simulation Summary")
print("=" * 80)

if stats['total_predictions'] > 0:
    print(f"Predictions processed: {stats['total_predictions']:,}")
    print(f"Ground truth updates created: {stats['updates_created']:,}")
    print(f"Actual fraud: {stats['actual_fraud']:,} ({stats['actual_fraud']/stats['total_predictions']*100:.1f}%)")
    print(f"False positives: {stats['false_positives']:,}")
    print(f"False negatives: {stats['false_negatives']:,}")
    print(f"Model accuracy: {stats.get('accuracy', GROUND_TRUTH_SIM_ACCURACY)*100:.1f}%")
    print("=" * 80)
    print("\n✅ Ground truth simulation complete!")
    print("   Next: Run the cell below to apply updates to inference_responses table")
else:
    print("⚠️  No predictions found to simulate ground truth")
    print("   Total predictions: 0")
    print("   Possible reasons:")
    print("   - No inference tests have been run yet (run Cells 6-9)")
    print("   - All predictions already have ground truth")
    print("   - Wrong endpoint name filter")
    print("\n   Action: Run Cells 6-9 to generate inference predictions first")
    print("=" * 80)


Waiting 60s for the final SQS→Lambda flush cycle...


2026-06-26 21:06:23,995 - INFO - Initialized GroundTruthSimulator:


2026-06-26 21:06:23,996 - INFO -   Base accuracy        : 0.85


2026-06-26 21:06:23,996 - INFO -   Effective accuracy   : 0.85


2026-06-26 21:06:23,997 - INFO - ================================================================================


...

2026-06-26 21:06:34,386 - INFO -   False positives: 0


2026-06-26 21:06:34,387 - INFO -   False negatives: 15


2026-06-26 21:06:34,387 - INFO -   Model accuracy: 85.0%


2026-06-26 21:06:34,388 - INFO - ================================================================================



Simulation Summary
Predictions processed: 102
Ground truth updates created: 102
Actual fraud: 15 (14.7%)
False positives: 0
False negatives: 15
Model accuracy: 85.0%

✅ Ground truth simulation complete!
   Next: Run the cell below to apply updates to inference_responses table


In [10]:
# Apply simulated ground truth updates to inference_responses table
print("Applying ground truth updates to inference_responses table...")

# Initialize updater (already imported above)
if 'updater' not in dir():
    from src.drift_monitoring.update_ground_truth import GroundTruthUpdater
    updater = GroundTruthUpdater(athena_client=athena_client, dry_run=False)

# Get statistics before update
print("\nGetting update statistics...")
update_stats = updater.get_update_statistics()

if update_stats['total_updates'] > 0:
    print(f"\nPending updates: {update_stats['total_updates']:,}")
    print(f"  Fraud cases: {update_stats['fraud_cases']:,}")
    print(f"  False positives: {update_stats['false_positives']:,}")
    print(f"  False negatives: {update_stats['false_negatives']:,}")
    print(f"  Avg days to confirmation: {update_stats['avg_days_to_confirmation']:.1f}")
    
    # Execute the update
    print("\nMerging ground truth updates into inference_responses...")
    result = updater.update_ground_truth_batch()
    
    print(f"\n✅ Ground truth update complete!")
    print(f"   Records updated: {result['updated']:,}")
    print(f"\n   Next: Run the cell below to check coverage")
else:
    print("\n⚠️  No pending updates found")
    print("   Make sure you ran the simulation cell above first")

2026-06-26 21:06:34,403 - INFO - Calculating update statistics...


2026-06-26 21:06:34,403 - INFO - Executing query: 
        SELECT
            COUNT(*) as total_updates,
            COALESCE(SUM(CAST(gtu.actual_frau...


Applying ground truth updates to inference_responses table...

Getting update statistics...


2026-06-26 21:06:36,867 - INFO - Query returned 1 rows


2026-06-26 21:06:36,868 - INFO - ================================================================================


...

2026-06-26 21:06:39,346 - INFO - 
Executing MERGE statement...


2026-06-26 21:06:39,346 - INFO - Executing query: 
        MERGE INTO fraud_detection.inference_responses AS target
        USING (
            SELECT...


2026-06-26 21:06:43,672 - INFO - Query executed successfully


2026-06-26 21:06:43,673 - INFO - ✓ Successfully updated 102 records



✅ Ground truth update complete!
   Records updated: 102

   Next: Run the cell below to check coverage


### 4.3 Verify coverage

Counts how many `inference_responses` rows still lack `ground_truth`.
Should be 0 after Section 4.2 completes. If non-zero, those rows were
logged after the simulator ran — re-execute 4.2 to label them.


In [11]:
# Check current ground truth coverage before update
from src.drift_monitoring.update_ground_truth import GroundTruthUpdater

updater = GroundTruthUpdater(athena_client=athena_client, dry_run=False)

coverage_before = updater.get_coverage_statistics()
print('Current ground truth coverage:')
print(f"  Total predictions: {coverage_before['total_predictions']:,}")
print(f"  With ground truth: {coverage_before['with_ground_truth']:,} ({coverage_before['coverage_pct']:.2f}%)")
print(f"  Without ground truth: {coverage_before['without_ground_truth']:,}")

2026-06-26 21:06:43,680 - INFO - Calculating ground truth coverage...


2026-06-26 21:06:43,681 - INFO - Executing query: 
        SELECT
            COUNT(*) as total_predictions,
            COALESCE(SUM(CASE WHEN ground...


2026-06-26 21:06:46,163 - INFO - Query returned 1 rows


Current ground truth coverage:
  Total predictions: 205
  With ground truth: 205 (100.00%)
  Without ground truth: 0


## 5. Monitoring Pipeline Setup

CloudFormation creates the inference-logging path (already done). This section wires up the *monitoring-results writer* + drift-monitor Lambda permissions. Idempotent — safe to re-run.


### 5.1 Lake Formation grants

In [ ]:
# Grant Lake Formation permissions if LF is in managed mode (no-op otherwise).
# Details: src/setup/grant_lake_formation_permissions.py
from src.setup.grant_lake_formation_permissions import grant_lake_formation_permissions

grant_lake_formation_permissions(
    database=ATHENA_DATABASE,
    region=REGION,
    lambda_role_arn=LAMBDA_EXEC_ROLE,
    monitoring_table=MONITORING_TABLE_NAME,
)


### 5.2 Deploy monitoring-results writer Lambda

The drift Lambda emits one SQS message per monitoring run. A writer Lambda consumes the queue and INSERTs into `monitoring_responses`. Both are created here (CFN intentionally delegates this).


In [13]:
# Lambda Deployment Control.
# False (default): the deploy cells below skip if the Lambda already exists.
#                  Safe to re-run the notebook top-to-bottom without churn.
# True:            force redeploy — use after editing Lambda source code
#                  (src/drift_monitoring/lambda_drift_monitor.py or
#                  src/drift_monitoring/deploy_monitoring_writer.py).
REDEPLOY_LAMBDAS = True

print(f"Lambda Redeployment: {'ENABLED (force redeploy)' if REDEPLOY_LAMBDAS else 'DISABLED (skip if exists)'}")


Lambda Redeployment: ENABLED (force redeploy)


In [ ]:
# Bootstrap the drift-monitor Lambda's IAM role + inline policies.
#
# Why this lives in the notebook (and NOT in CloudFormation):
#   1. The role name and Lambda function name come from runtime config
#      (LAMBDA_EXEC_ROLE in .env) that varies per developer / per environment.
#      CFN parameters could carry them, but it's brittle when the .env is the
#      source of truth.
#   2. The drift-monitor Lambda itself is created later by
#      scripts/deploy_lambda_container.sh (Section 7.2) — a container image
#      push that CFN can't model (CFN cannot run `docker build`). Keeping the
#      role bootstrap in the same place as the function deployment keeps the
#      "create role → create function" pair together instead of split between
#      a CFN template and a notebook.
#   3. This block is idempotent: get_role → if missing, create_role → re-attach
#      managed policies → re-put inline policy. Safe to re-run on every kernel
#      session.

import boto3, json, time
from src.config.config import LAMBDA_EXEC_ROLE

iam = boto3.client('iam', region_name=REGION)
ROLE_NAME = LAMBDA_EXEC_ROLE.split('/')[-1] if LAMBDA_EXEC_ROLE else 'fraud-detection-drift-monitor-role'

trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'lambda.amazonaws.com'},
        'Action': 'sts:AssumeRole',
    }],
}

# Create role if it doesn't exist
try:
    iam.get_role(RoleName=ROLE_NAME)
    print(f'\u2713 Role {ROLE_NAME} already exists')
except iam.exceptions.NoSuchEntityException:
    iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Lambda role for fraud detection drift monitoring',
    )
    print(f'\u2713 Created role {ROLE_NAME}')
    time.sleep(5)  # Wait for IAM propagation

# Attach managed policies
managed_policies = [
    'arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole',
    'arn:aws:iam::aws:policy/AmazonAthenaFullAccess',
]
for policy_arn in managed_policies:
    try:
        iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn=policy_arn)
        print(f'\u2713 Attached {policy_arn.split("/")[-1]}')
    except Exception as e:
        print(f'  (already attached or error: {e})')

# Add inline policy for SQS + S3 access
inline_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Action': [
                'sqs:ReceiveMessage',
                'sqs:DeleteMessage',
                'sqs:GetQueueAttributes',
            ],
            'Resource': '*',
        },
        {
            'Effect': 'Allow',
            'Action': [
                's3:GetObject',
                's3:PutObject',
                's3:ListBucket',
                's3:GetBucketLocation',
            ],
            'Resource': '*',
        },
    ],
}
iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName='SQS-S3-Access',
    PolicyDocument=json.dumps(inline_policy),
)
print(f'\u2713 Attached inline SQS + S3 policy')

print(f'\n\u2713 Role ready: {ROLE_NAME}')
print(f'  Waiting 10s for IAM propagation...')
time.sleep(10)

In [15]:
# Deploy monitoring results writer Lambda
import boto3
import subprocess

lambda_client = boto3.client('lambda', region_name=REGION)
WRITER_NAME = MONITORING_WRITER_LAMBDA

should_deploy = REDEPLOY_LAMBDAS
if not REDEPLOY_LAMBDAS:
    try:
        lambda_client.get_function(FunctionName=WRITER_NAME)
        print(f"✓ Lambda '{WRITER_NAME}' exists. Skipping deployment.")
        print("  (Set REDEPLOY_LAMBDAS = True to force redeployment)")
    except lambda_client.exceptions.ResourceNotFoundException:
        print(f"Lambda '{WRITER_NAME}' not found. Proceeding with deployment...")
        should_deploy = True
else:
    print("REDEPLOY_LAMBDAS enabled. Forcing redeployment...")

if should_deploy:
    print("=" * 80)
    result = subprocess.run(
        ['python3', '-m', 'src.drift_monitoring.deploy_monitoring_writer', REGION],
        cwd=project_root,
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr)
    else:
        print("\n✓ Monitoring writer deployed!")

# Get queue URL for drift monitor
sqs = boto3.client('sqs', region_name=REGION)
try:
    queue_url = sqs.get_queue_url(QueueName=MONITORING_SQS_QUEUE_NAME)['QueueUrl']
    print(f"\nSQS Queue URL: {queue_url}")
    print("(This will be used by the drift monitor Lambda)")
except Exception as e:
    print(f"Note: {e}")


REDEPLOY_LAMBDAS enabled. Forcing redeployment...


╔════════════════════════════════════════════════════════════════════╗
║  Deploying Monitoring Results Writer Lambda                       ║
╚════════════════════════════════════════════════════════════════════╝

  Region: us-west-2
  Account: <ACCOUNT_ID>
  Lambda: fraud-monitoring-results-writer
  Queue: fraud-monitoring-results

[1/5] Creating SQS queue...
  ✓ Queue created: https://sqs.us-west-2.amazonaws.com/<ACCOUNT_ID>/fraud-monitoring-results

[2/5] Creating IAM role...
  ✓ Role exists
  ✓ Policies attached
  Waiting 10s for role propagation...

[3/5] Creating Lambda package...
  ✓ Package created: 1.5 KB

[4/5] Deploying Lambda function...
  ✓ Lambda updated

[5/5] Configuring SQS trigger...
  ✓ SQS trigger exists

╔════════════════════════════════════════════════════════════════════╗
║  ✅ MONITORING WRITER DEPLOYED                                     ║
╚════════════════════════════════════════════════════════════════════╝

  Lambda: arn:aws:lambda:us-west-2:<ACCOUNT_ID>:funct

### 5.3 Wire SQS queue URL into drift-monitor Lambda

Only relevant if the drift-monitor Lambda was already created in a previous run. Skipped silently otherwise; Section 7.2 creates it correctly the first time.


In [16]:
import boto3, time, os

lam = boto3.client('lambda', region_name=REGION)
DRIFT_LAMBDA = os.getenv('DRIFT_LAMBDA_NAME', DRIFT_LAMBDA_NAME)

try:
    config = lam.get_function_configuration(FunctionName=DRIFT_LAMBDA)
    env_vars = config.get('Environment', {}).get('Variables', {})
    env_vars['MONITORING_SQS_QUEUE_URL'] = queue_url

    # Wait for any pending updates
    time.sleep(2)
    lam.update_function_configuration(
        FunctionName=DRIFT_LAMBDA,
        Environment={'Variables': env_vars},
    )
    print(f'\u2713 Added MONITORING_SQS_QUEUE_URL to {DRIFT_LAMBDA}')
    print(f'  Queue URL: {queue_url}')
except Exception as e:
    print(f'Note: Could not update {DRIFT_LAMBDA}: {e}')
    print(f'  Lambda will be created with correct env var in Section 7.2')

Note: Could not update fraud-detection-drift-monitor: An error occurred (ResourceNotFoundException) when calling the GetFunctionConfiguration operation: Function not found: arn:aws:lambda:us-west-2:<ACCOUNT_ID>:function:fraud-detection-drift-monitor
  Lambda will be created with correct env var in Section 7.2


## 6. Drift Detection

Compares production traffic against the **frozen `evaluation_data`
Iceberg snapshot** — the held-out test set the model was scored on at
training time. Lineage info pinned in baseline.json (registered on the
ModelPackage) makes each monitoring row in Athena traceable to:
  - the exact endpoint that produced the predictions
  - the model package ARN serving those predictions
  - the Iceberg snapshot ID that defined the baseline

Two checks:
- **6.2 Data drift**: per-feature KS test, `evaluation_data.X` vs `inference_responses.input_features.X`
- **6.3 Model drift**: classification metrics, `evaluation_data.(is_fraud, fraud_prediction)` vs `inference_responses.(ground_truth, prediction)`


### 6.0 Run setup — generate `monitoring_run_id` + resolve baseline lineage

Each pass through Section 6 produces one `monitoring_run_id`. The same id
is used by the data-drift cell (6.2), the model-drift cell (6.3), and the
two writes in 6.4:

- INSERT one row into `monitoring_responses` keyed on this id
- UPDATE `inference_responses` to backfill `monitoring_run_id` on every
  prediction that this run scored — so QuickSight can join the two tables

This cell also looks up `MAX(monitoring_timestamp)` from prior runs. The
drift cells below use it to scope the "current" window to **only predictions
since the last monitoring run** — so re-running Section 2/3/6 measures only
the new drift, not all cumulative inferences.


In [17]:
# Resolve run id + lineage + the "since when" cutoff for the current window.
from src.drift_monitoring.lambda_drift_monitor import load_baseline_from_registry
from datetime import datetime as _dt

monitoring_run_id = f'notebook-drift-{_dt.now().strftime("%Y%m%d-%H%M%S")}'
monitoring_run_ts = _dt.now()

# --- Baseline lineage (single lookup, reused by 6.2 / 6.3 / 6.4) ---
_baseline_meta = load_baseline_from_registry() or {}
model_package_arn      = _baseline_meta.get('model_package_arn') or '(not pinned)'
training_table         = _baseline_meta.get('training_table')   or ATHENA_TRAINING_TABLE
training_snapshot_id   = _baseline_meta.get('training_snapshot_id') or ''
evaluation_table       = _baseline_meta.get('evaluation_table') or ATHENA_EVALUATION_TABLE
evaluation_snapshot_id = _baseline_meta.get('evaluation_snapshot_id') or ''

# --- Previous monitoring timestamp (delta filter for the "current" window) ---
prev_ts_query = f"""
    SELECT MAX(monitoring_timestamp) AS prev_ts
    FROM {ATHENA_DATABASE}.{MONITORING_TABLE_NAME}
    WHERE endpoint_name = '{ENDPOINT_NAME}'
"""
try:
    _prev = athena_client.execute_query(prev_ts_query)
    if _prev.empty:
        prev_monitoring_ts = None
    else:
        _val = _prev['prev_ts'].iloc[0]
        # pandas converts SQL NULL TIMESTAMP -> pd.NaT, which is NOT None
        # but ALSO can't be formatted as a TIMESTAMP literal. Treat both as cold-start.
        prev_monitoring_ts = None if pd.isna(_val) else _val
except Exception:
    # First-ever run: table empty / not yet readable. Cold-start fallback.
    prev_monitoring_ts = None

def _from(table, snapshot_id):
    return (f'{ATHENA_DATABASE}.{table} FOR VERSION AS OF {snapshot_id}'
            if snapshot_id else f'{ATHENA_DATABASE}.{table}')

def _snap_log(snapshot_id):
    return f'Iceberg snapshot {snapshot_id}' if snapshot_id else 'live table (no snapshot pinned)'

print('━' * 72)
print('MONITORING RUN — LINEAGE')
print('━' * 72)
print(f'  monitoring_run_id   : {monitoring_run_id}')
print(f'  Endpoint            : {ENDPOINT_NAME}')
print(f'  Model package ARN   : {model_package_arn}')
print(f'  Training baseline   : {ATHENA_DATABASE}.{training_table}  ({_snap_log(training_snapshot_id)})')
print(f'  Evaluation baseline : {ATHENA_DATABASE}.{evaluation_table}  ({_snap_log(evaluation_snapshot_id)})')
if prev_monitoring_ts is not None:
    print(f'  Current window      : predictions since last run @ {prev_monitoring_ts}')
else:
    print(f'  Current window      : cold-start fallback (last {MONITORING_DATA_DRIFT_LOOKBACK_DAYS} days for data drift, '
          f'last 30 days for model drift)')
print('━' * 72)


ModuleNotFoundError: No module named 'evidently'

### 6.1 Overall classification metrics

`ModelPerformanceMonitor` queries `inference_responses` directly and
computes precision/recall/F1/ROC-AUC/PR-AUC against `ground_truth`.
This is the "current" half of the model-drift comparison — printed
inline so you can spot trouble before the more detailed Evidently
report in 6.3.


In [ ]:
from src.drift_monitoring.monitor_model_performance import ModelPerformanceMonitor

monitor = ModelPerformanceMonitor(
    athena_client=athena_client,
    alert_threshold=0.85,  # ROC-AUC threshold for alerts
    min_samples=50,        # Minimum samples for reliable metrics
)

# Check ground truth coverage
coverage = monitor.get_ground_truth_coverage(endpoint_name=ENDPOINT_NAME, days=30)
print(f"Ground truth coverage (last 30 days):")
print(f"  Total predictions: {coverage['total_predictions']:,}")
print(f"  With ground truth: {coverage['with_ground_truth']:,} ({coverage['coverage_pct']:.2f}%)")

In [ ]:
# Generate performance report
report = monitor.generate_performance_report(
    endpoint_name=ENDPOINT_NAME,
    days=7,
    window='D',  # Daily time windows
)

monitor.print_report_summary(report)

In [ ]:
# Display overall metrics
if report.get('overall_metrics') and 'error' not in report['overall_metrics']:
    metrics = report['overall_metrics']
    print('Overall Model Performance:')
    print(f"  ROC-AUC:    {metrics.get('roc_auc', 'N/A')}")
    print(f"  PR-AUC:     {metrics.get('pr_auc', 'N/A')}")
    print(f"  Precision:  {metrics['precision']:.4f}")
    print(f"  Recall:     {metrics['recall']:.4f}")
    print(f"  F1-Score:   {metrics['f1_score']:.4f}")
    print(f"  Accuracy:   {metrics['accuracy']:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  TP: {metrics['true_positives']:,}  FP: {metrics['false_positives']:,}")
    print(f"  FN: {metrics['false_negatives']:,}  TN: {metrics['true_negatives']:,}")
else:
    print('Insufficient ground truth data for metrics. Run ground truth update first.')

### 6.2 Data drift

Per-feature Kolmogorov-Smirnov test (Evidently `DataDriftPreset`).
Baseline rows come from `evaluation_data` pinned to a specific Iceberg
snapshot via `baseline.json` on the registered ModelPackage. Current
rows come from `inference_responses` within the configured lookback
window (`MONITORING_DATA_DRIFT_LOOKBACK_DAYS` in config.yaml).

The lineage banner at the top of the next cell shows which exact
baseline and which current window were used.


In [ ]:
# (imports moved into the data-drift cell below)


In [ ]:
import json as _json
from src.drift_monitoring.evidently_reports import run_data_drift_report

# ════════════════════════════════════════════════════════════════════════
# DATA DRIFT — feature-distribution shift (Evidently DataDriftPreset)
# ════════════════════════════════════════════════════════════════════════
# Baseline : `training_data` Iceberg slice (the distribution the model was
#            TRAINED on — this is the industry-standard reference for data
#            drift: did production traffic move away from what we taught it?).
#            Lambda also uses training_data after this redesign.
# Current  : `inference_responses` rows since the last monitoring run
#            (or the last MONITORING_DATA_DRIFT_LOOKBACK_DAYS as cold-start).
# Method   : per-feature KS-test (numerical), p-value < 0.05 → drifted.
# Excluded : `customer_gender` (STRING in training_data, encoded float in
#            inference_responses — can't run KS across the type mismatch).

BASELINE_LIMIT = 5000
INFERENCE_LIMIT = 10000
NUMERIC_FEATURES = [f for f in TRAINING_FEATURES if f != 'customer_gender']

baseline_from = _from(training_table, training_snapshot_id)

# Current window: since last monitoring run if available, else cold-start lookback
if prev_monitoring_ts is not None:
    current_filter = f"request_timestamp > TIMESTAMP '{prev_monitoring_ts}'"
    window_label = f'since last monitoring run @ {prev_monitoring_ts}'
else:
    cold_start = (_dt.now() - timedelta(days=MONITORING_DATA_DRIFT_LOOKBACK_DAYS)).strftime('%Y-%m-%d %H:%M:%S')
    current_filter = f"request_timestamp >= TIMESTAMP '{cold_start}'"
    window_label = f'cold-start fallback (last {MONITORING_DATA_DRIFT_LOOKBACK_DAYS} days)'

print(f'Baseline source     : {ATHENA_DATABASE}.{training_table}  ({_snap_log(training_snapshot_id)})')
print(f'Current window      : {window_label}')
print(f'Features compared   : {len(NUMERIC_FEATURES)} numeric (customer_gender excluded — type mismatch)')

# --- Baseline rows ---
baseline_sql = f"""
    SELECT {", ".join(NUMERIC_FEATURES)}
    FROM {baseline_from}
    WHERE is_fraud IS NOT NULL
    LIMIT {BASELINE_LIMIT}
"""
baseline_df = athena_client.execute_query(baseline_sql)
baseline_df = baseline_df.apply(pd.to_numeric, errors='coerce').dropna(how='all')
print(f'\nBaseline : {len(baseline_df):,} rows × {len(NUMERIC_FEATURES)} features')

# --- Current rows (parse JSON-encoded features) ---
inference_sql = f"""
    SELECT input_features
    FROM {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
    WHERE endpoint_name = '{ENDPOINT_NAME}'
      AND {current_filter}
    LIMIT {INFERENCE_LIMIT}
"""
inf_raw = athena_client.execute_query(inference_sql)
print(f'Current  : {len(inf_raw):,} rows')

drift_results = None
if inf_raw.empty:
    print('\n⚠ No new inference data in the current window. Run Section 3 again, then re-run Section 6.')
elif len(inf_raw) < 50:
    print(f'\n⚠ Only {len(inf_raw)} rows — statistical tests need ≥ 50 for reliable p-values. Send more.')
else:
    current_df = pd.DataFrame([
        {f: float(d.get(f, float("nan"))) for f in NUMERIC_FEATURES}
        for d in (_json.loads(r) for r in inf_raw['input_features'])
    ]).dropna(how='all')

    valid = [c for c in NUMERIC_FEATURES
             if c in baseline_df.columns and c in current_df.columns
             and baseline_df[c].notna().any() and current_df[c].notna().any()]
    if len(valid) < len(NUMERIC_FEATURES):
        dropped = set(NUMERIC_FEATURES) - set(valid)
        print(f'  ⚠ Dropping {len(dropped)} all-NaN columns: {sorted(dropped)}')

    drift_results = run_data_drift_report(
        baseline_df=baseline_df[valid],
        current_df=current_df[valid],
    )

    print('\nRESULTS')
    print('━' * 72)
    print(f'  Drift detected     : {drift_results["drift_detected"]}')
    print(f'  Drifted features   : {drift_results["drifted_columns_count"]} / {len(valid)}'
          f' ({drift_results["drifted_columns_share"]:.1%})')
    if drift_results["drift_detected"]:
        top = sorted(
            ((c, info["drift_score"]) for c, info in drift_results.get("per_column", {}).items() if info.get("drifted")),
            key=lambda x: x[1],
        )[:5]
        print('  Top 5 drifted      : ' + ', '.join(f'{c} (p={s:.4f})' for c, s in top))
    print('━' * 72)


In [ ]:
# Display the interactive Evidently DataDriftPreset report.
if 'drift_results' in dir() and drift_results and drift_results.get('snapshot'):
    print('📊 Evidently Data Drift Report (interactive)')
    print('=' * 80)
    display(drift_results['snapshot'])
else:
    print('No drift results to visualize — re-run the cell above.')


In [ ]:
# Per-feature KS p-values, sorted ascending (lower = more drifted).
if 'drift_results' in dir() and drift_results and drift_results.get('per_column'):
    print('📋 Per-Column Drift Summary')
    print('=' * 80)
    for col, info in sorted(drift_results['per_column'].items(), key=lambda x: x[1]['drift_score']):
        status = '⚠ DRIFT' if info['drifted'] else '✓ OK'
        print(f'  {col:35s} p-value={info["drift_score"]:.4f}  [{status}]')
else:
    print('No drift results — re-run the cell above.')


### 6.3 Model drift

Classification-metric drift (Evidently `ClassificationPreset`).
Baseline `(target, prediction)` pairs come from `evaluation_data`
(the model's scored test set, frozen at training time). Current
`(ground_truth, prediction)` pairs come from `inference_responses`
where Section 4 has attached labels.

If you see "Insufficient ground truth", run Section 4.2 (simulator) again.


In [ ]:
from src.drift_monitoring.evidently_reports import run_classification_report

# ════════════════════════════════════════════════════════════════════════
# MODEL DRIFT — Evidently ClassificationPreset
# ════════════════════════════════════════════════════════════════════════
# Baseline : (is_fraud, fraud_prediction) from `evaluation_data` (the
#            labeled held-out test set the model was scored on at training
#            time — this is the right reference for model-drift, since
#            training_data labels are what the model already absorbed).
# Current  : (ground_truth, prediction) from `inference_responses` since
#            the last monitoring run (or last 30 days cold-start).
#
# Both datasets must contain BOTH classes (0 and 1). evaluation_data is
# ~0.2% fraud, so a flat LIMIT typically returns only class 0 → we
# stratify the baseline pull below to guarantee both labels appear.
# (See evidently_reports.run_classification_report for the bug context.)

MIN_SAMPLES = 50
MODEL_DRIFT_LOOKBACK_DAYS = 30
PREDICTION_THRESHOLD = 0.5

baseline_from = _from(evaluation_table, evaluation_snapshot_id)

# Current window: since last monitoring run, else cold-start
if prev_monitoring_ts is not None:
    days_window = MODEL_DRIFT_LOOKBACK_DAYS  # caps the load_predictions_with_ground_truth window
    window_label = f'since last monitoring run @ {prev_monitoring_ts}'
else:
    days_window = MODEL_DRIFT_LOOKBACK_DAYS
    window_label = f'cold-start fallback (last {MODEL_DRIFT_LOOKBACK_DAYS} days)'

print(f'Baseline source     : {ATHENA_DATABASE}.{evaluation_table}  ({_snap_log(evaluation_snapshot_id)})')
print(f'Current window      : {window_label}')
print(f'Decision threshold  : probability_fraud >= {PREDICTION_THRESHOLD}')

# Pull current predictions with ground truth, then apply the delta filter manually.
gt_df = monitor.load_predictions_with_ground_truth(
    endpoint_name=ENDPOINT_NAME, days=days_window,
)
if prev_monitoring_ts is not None and not gt_df.empty:
    gt_df = gt_df[gt_df['request_timestamp'] > prev_monitoring_ts]
print(f'\nCurrent  : {len(gt_df):,} predictions with ground truth')

model_report = None
if len(gt_df) < MIN_SAMPLES:
    print(f'⚠ Need ≥ {MIN_SAMPLES} predictions with ground truth — run Section 4.2 (simulator) again.')
elif gt_df['ground_truth'].nunique() < 2:
    print(f'⚠ Current sample has only one class — increase simulator accuracy degradation or send more drifted samples.')
else:
    current_df = pd.DataFrame({
        'target':     gt_df['ground_truth'].astype(int).values,
        'prediction': (gt_df['probability_fraud'] >= PREDICTION_THRESHOLD).astype(int).values,
    })

    # Stratified baseline pull
    half = max(MIN_SAMPLES // 2, len(gt_df) // 2)
    baseline_sql = f"""
        (SELECT CAST(is_fraud AS INT) AS target,
                CAST(fraud_prediction AS INT) AS prediction
         FROM {baseline_from}
         WHERE is_fraud = TRUE AND fraud_prediction IS NOT NULL
         LIMIT {half})
        UNION ALL
        (SELECT CAST(is_fraud AS INT) AS target,
                CAST(fraud_prediction AS INT) AS prediction
         FROM {baseline_from}
         WHERE is_fraud = FALSE AND fraud_prediction IS NOT NULL
         LIMIT {half})
    """
    baseline_df = athena_client.execute_query(baseline_sql)
    print(f'Baseline : {len(baseline_df):,} predictions '
          f'(stratified: {sum(baseline_df["target"] == 1)} fraud / {sum(baseline_df["target"] == 0)} non-fraud)')

    # Evidently ClassificationPreset needs BOTH classes in BOTH target AND
    # prediction of BOTH datasets. Check all four before calling; print a
    # clear skip message if any side is degenerate (model never predicted
    # the minority class, etc.).
    sides = [('baseline.target', baseline_df['target']),
             ('baseline.prediction', baseline_df['prediction']),
             ('current.target', current_df['target']),
             ('current.prediction', current_df['prediction'])]
    degenerate = [name for name, col in sides if col.nunique() < 2]
    if degenerate:
        print(f'⚠ Skipping classification report — single-class column(s): {degenerate}')
        print(f'  Likely cause: model never predicted the minority class on this sample.')
        print(f'  Fix: send more drifted inferences, or lower PREDICTION_THRESHOLD below 0.5.')
    else:
        model_report = run_classification_report(
            baseline_df=baseline_df, current_df=current_df,
            target_column='target', prediction_column='prediction',
        )
        print('\n✓ Classification report ready — interactive view below.')

if model_report:
    print()
    print('Evidently Classification Report (interactive)')
    print('=' * 80)
    display(model_report['snapshot'])


### 6.4 Log monitoring results to MLflow and Athena

Two writes:
1. **MLflow**: one run under the `credit-card-fraud-detection-monitoring`
   experiment with all drift metrics + the HTML reports as artifacts.
2. **Athena `monitoring_responses`**: one row with `model_package_arn` +
   `evaluation_snapshot_id` so this monitoring run is fully traceable
   alongside the daily Lambda-produced rows in QuickSight.


In [ ]:
from src.drift_monitoring.log_monitoring_to_mlflow import log_monitoring_to_mlflow

drift_data   = drift_results if 'drift_results' in dir() else None
model_data   = model_report  if 'model_report'  in dir() else None
metrics_data = report.get('overall_metrics') if 'report' in dir() and report.get('overall_metrics') else None

result = log_monitoring_to_mlflow(
    drift_results=drift_data,
    model_report=model_data,
    overall_metrics=metrics_data,
    endpoint_name=ENDPOINT_NAME,
    model_name=MLFLOW_MODEL_NAME,
    region=REGION,
)
if result['success']:
    print(f"✓ Logged to MLflow — run_id={result['mlflow_run_id']}")
else:
    print(f"⚠ MLflow logging failed: {result.get('error', 'unknown')}")


In [ ]:
# Write the monitoring run + tag the inference rows it measured.
#
# Two Athena ops, both keyed on the SAME monitoring_run_id (resolved in 6.0):
#   1. INSERT one row into monitoring_responses (the drift verdict for this run)
#   2. UPDATE inference_responses to backfill monitoring_run_id on the
#      predictions that fell in this run's current window
#
# Column list in op #1 MUST match the CFN DDL
# (cloudformation/sagemaker-mlflow-setup.yaml: monitoring_responses) AND
# src/drift_monitoring/deploy_monitoring_writer.py. Keep all three in sync.
import json as _json

# --- Aggregate drift findings into Athena-bound values ---
data_detected, drifted_count, drifted_share = False, 0, 0.0
features_analyzed, data_sample = 0, 0
per_feature = {}
if 'drift_results' in dir() and drift_results:
    data_detected     = drift_results.get('drift_detected', False)
    drifted_count     = drift_results.get('drifted_columns_count', 0)
    drifted_share     = drift_results.get('drifted_columns_share', 0)
    features_analyzed = drift_results.get('features_analyzed', 0)
    data_sample       = drift_results.get('sample_size', 0)
    for col, info in drift_results.get('per_column', {}).items():
        per_feature[col] = info.get('drift_score', 0)

mdrift = baseline_auc = current_auc = degrad = degrad_pct = None
acc = prec = rec = f1 = model_sample = None
if 'report' in dir() and report.get('overall_metrics') and 'error' not in report.get('overall_metrics', {}):
    om = report['overall_metrics']
    mdrift       = om.get('model_drift_detected', False)
    baseline_auc = om.get('baseline_roc_auc')
    current_auc  = om.get('current_roc_auc')
    degrad       = om.get('roc_auc_degradation')
    degrad_pct   = om.get('roc_auc_degradation_pct')
    acc          = om.get('accuracy')
    prec         = om.get('precision')
    rec          = om.get('recall')
    if prec and rec and (prec + rec) > 0:
        f1 = 2 * prec * rec / (prec + rec)
    model_sample = om.get('sample_size')

def sql_val(v):
    if v is None:                return 'NULL'
    if isinstance(v, bool):      return 'TRUE' if v else 'FALSE'
    if isinstance(v, str):       return "'" + v.replace("'", "''") + "'"
    return str(v)

mp_arn = _baseline_meta.get('model_package_arn')
# Use the model-package short-id as the model_version label so it's readable
model_version_label = mp_arn.split('/')[-1] if mp_arn else 'latest'

columns = [
    'monitoring_run_id', 'monitoring_timestamp',
    'endpoint_name', 'model_version', 'model_package_arn', 'evaluation_snapshot_id',
    'data_drift_detected', 'drifted_columns_count', 'drifted_columns_share',
    'features_analyzed', 'data_sample_size', 'model_drift_detected',
    'baseline_roc_auc', 'current_roc_auc',
    'roc_auc_degradation', 'roc_auc_degradation_pct',
    'accuracy', 'precision', 'recall', 'f1_score',
    'model_sample_size', 'per_feature_drift_scores',
    'evidently_report_s3_path', 'mlflow_run_id',
    'alert_sent', 'detection_engine', 'created_at',
]
ts = f"TIMESTAMP '{monitoring_run_ts.strftime('%Y-%m-%d %H:%M:%S')}'"
values = [
    sql_val(monitoring_run_id),
    ts,
    sql_val(ENDPOINT_NAME),
    sql_val(model_version_label),
    sql_val(mp_arn),
    sql_val(evaluation_snapshot_id or None),
    sql_val(data_detected), sql_val(drifted_count), sql_val(drifted_share),
    sql_val(features_analyzed), sql_val(data_sample), sql_val(mdrift),
    sql_val(baseline_auc), sql_val(current_auc),
    sql_val(degrad), sql_val(degrad_pct),
    sql_val(acc), sql_val(prec), sql_val(rec), sql_val(f1),
    sql_val(model_sample),
    sql_val(_json.dumps(per_feature) if per_feature else None),
    sql_val(None),                                  # evidently_report_s3_path
    sql_val(result.get('mlflow_run_id') if 'result' in dir() and isinstance(result, dict) else None),
    sql_val(data_detected or mdrift),
    sql_val('evidently'),
    ts,
]
insert_sql = (
    f"INSERT INTO {ATHENA_DATABASE}.{MONITORING_TABLE_NAME} "
    f"({', '.join(columns)}) VALUES ({', '.join(values)})"
)

# --- 1) Write the monitoring_responses row ---
try:
    athena_client.execute_query(insert_sql, return_results=False)
    print(f'✓ Wrote {monitoring_run_id} to {ATHENA_DATABASE}.{MONITORING_TABLE_NAME}')
    print(f'  Model package ARN     : {mp_arn or "(not pinned)"}')
    print(f'  Evaluation snapshot id: {evaluation_snapshot_id or "(not pinned)"}')
    print(f'  Training snapshot id  : {training_snapshot_id or "(not pinned)"}')
    print(f'  Data drift detected   : {data_detected}')
    print(f'  Model drift detected  : {mdrift}')
except Exception as e:
    print(f'✗ Failed to write monitoring_responses row: {e}')

# --- 2) Backfill monitoring_run_id on the inference rows we just measured ---
# Window is the same one cells 6.2 / 6.3 used: (prev_monitoring_ts, monitoring_run_ts].
# UPDATE only touches rows still tagged NULL so re-runs of this cell are idempotent.
window_lower = (
    f"TIMESTAMP '{prev_monitoring_ts}'" if prev_monitoring_ts is not None
    else f"TIMESTAMP '{(_dt.now() - timedelta(days=MONITORING_DATA_DRIFT_LOOKBACK_DAYS)).strftime('%Y-%m-%d %H:%M:%S')}'"
)
window_upper = f"TIMESTAMP '{monitoring_run_ts.strftime('%Y-%m-%d %H:%M:%S')}'"

backfill_sql = f"""
    UPDATE {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
    SET monitoring_run_id = '{monitoring_run_id}'
    WHERE endpoint_name = '{ENDPOINT_NAME}'
      AND monitoring_run_id IS NULL
      AND request_timestamp > {window_lower}
      AND request_timestamp <= {window_upper}
"""
try:
    athena_client.execute_query(backfill_sql, return_results=False)
    # Read back how many got tagged
    verify_sql = f"""
        SELECT COUNT(*) AS n FROM {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
        WHERE monitoring_run_id = '{monitoring_run_id}'
    """
    res = athena_client.execute_query(verify_sql)
    n_tagged = int(res['n'].iloc[0]) if not res.empty else 0
    print(f'✓ Tagged {n_tagged} inference rows with monitoring_run_id={monitoring_run_id}')
except Exception as e:
    print(f'⚠ Could not backfill monitoring_run_id on inference_responses: {e}')
    print('  (Schema may not include the column yet — see CFN inference_responses DDL.)')


### 6.5 Performance alerts (from this run)

`ModelPerformanceMonitor` flags daily/weekly buckets where ROC-AUC drops
below `MIN_ROC_AUC_THRESHOLD` (config.yaml). These are derived from the
6.1 report — purely informational here; the daily Lambda fires real
SNS alerts on the same data.


In [ ]:
# Check for performance degradation alerts
alerts = report.get('alerts', [])

if alerts:
    print(f'⚠ {len(alerts)} performance alert(s) detected:\n')
    for alert in alerts:
        print(f"  [{alert['severity'].upper()}] Period: {alert['period']}")
        print(f"    ROC-AUC: {alert['value']:.4f} (baseline: {alert['baseline']:.4f})")
        print(f"    Degradation: {alert['degradation_pct']:.1f}%")
        print(f"    Samples: {alert['sample_count']:,}")
        print()
else:
    print('✓ No performance degradation alerts. Model is performing within thresholds.')

### 6.6 CloudWatch dashboard + alarms (one-time setup)

Creates a CloudWatch dashboard plus alarms that fire when the daily Lambda emits a metric above `DATA_DRIFT_THRESHOLD` / `MODEL_DRIFT_THRESHOLD` (both in config.yaml). Idempotent — re-run to push threshold changes.


In [ ]:
# Create CloudWatch Dashboard + Alarms for drift monitoring.
# Thresholds come from src.config.config (sourced from config.yaml).
import subprocess
from src.config.config import (
    DATA_DRIFT_THRESHOLD as CFG_DATA_DRIFT_THRESHOLD,
    MODEL_DRIFT_THRESHOLD as CFG_MODEL_DRIFT_THRESHOLD,
)

print('Creating CloudWatch Dashboard and Alarms...')
result = subprocess.run(
    [
        'python3', '-m', 'src.drift_monitoring.create_cloudwatch_monitoring',
        '--region', REGION,
        '--endpoint', ENDPOINT_NAME,
        '--drift-threshold', str(CFG_MODEL_DRIFT_THRESHOLD),
        '--psi-threshold', str(CFG_DATA_DRIFT_THRESHOLD),
        '--evaluation-periods', '1',
    ],
    cwd=project_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)
else:
    print('✓ CloudWatch monitoring created.')


## 7. Schedule Automated Drift Monitoring

Deploys the EventBridge + Lambda + SNS chain that runs `lambda_drift_monitor` on the schedule defined in `config.yaml` → `drift_monitor.schedule`.


### 7.1 Alert configuration

In [ ]:
# Alert configuration. Email defaults to ALERT_EMAIL in .env;
# drift thresholds + schedule come from config.yaml (src.config.config).
from src.config.config import (
    DATA_DRIFT_THRESHOLD,
    MODEL_DRIFT_THRESHOLD,
    DRIFT_MONITOR_SCHEDULE,
)

ALERT_EMAIL = os.getenv('ALERT_EMAIL', '')
if not ALERT_EMAIL or '@' not in ALERT_EMAIL:
    print('⚠ ALERT_EMAIL not set in .env — drift alerts will not be subscribed.')
    print('  Set: echo "ALERT_EMAIL=you@example.com" >> .env')
else:
    print(f'✓ Alert email          : {ALERT_EMAIL}')

print(f'  Data drift threshold : PSI ≥ {DATA_DRIFT_THRESHOLD}')
print(f'  Model drift threshold: {MODEL_DRIFT_THRESHOLD * 100:.0f}% degradation')
print(f'  Schedule             : {DRIFT_MONITOR_SCHEDULE}')


### 7.2 Deploy drift-monitor Lambda (container image)

In [ ]:
# Deploy drift monitoring Lambda as container image
import subprocess
import os
import boto3

if not ALERT_EMAIL or ALERT_EMAIL == 'your-email@example.com':
    print("⚠️  Warning: ALERT_EMAIL not set!")
    print("Set it with: ALERT_EMAIL = 'your-email@example.com'")
    print("")

lambda_client = boto3.client('lambda', region_name=REGION)

should_deploy = REDEPLOY_LAMBDAS
if not REDEPLOY_LAMBDAS:
    try:
        lambda_client.get_function(FunctionName=DRIFT_LAMBDA_NAME)
        print(f"✓ Lambda '{DRIFT_LAMBDA_NAME}' exists. Skipping deployment.")
        print("  (Set REDEPLOY_LAMBDAS = True to force redeployment)")
    except lambda_client.exceptions.ResourceNotFoundException:
        print(f"Lambda '{DRIFT_LAMBDA_NAME}' not found. Proceeding with deployment...")
        should_deploy = True
else:
    print("REDEPLOY_LAMBDAS enabled. Forcing redeployment...")

if should_deploy:
    print("")
    print("╔════════════════════════════════════════════════════════════════════╗")
    print("║  Deploying Drift Monitoring Lambda (Container Image)              ║")
    print("╚════════════════════════════════════════════════════════════════════╝")
    print(f"  Email: {ALERT_EMAIL}")
    print(f"  Data Drift Threshold: {DATA_DRIFT_THRESHOLD}")
    print(f"  Model Drift Threshold: {MODEL_DRIFT_THRESHOLD}")
    print("")

    result = subprocess.run(
        ['bash', 'scripts/deploy_lambda_container.sh',
         ALERT_EMAIL,
         str(DATA_DRIFT_THRESHOLD),
         str(MODEL_DRIFT_THRESHOLD)],
        cwd=project_root,
        env={**os.environ, 'AWS_REGION': REGION},
        capture_output=False,
        text=True,
    )

    if result.returncode != 0:
        print("\n❌ Deployment failed!")
        print("Check the output above for errors")
    else:
        print("\n✅ Deployment successful!")
        print("\nNext steps:")
        print("  1. Check your email for SNS subscription confirmation")
        print(f"  2. Monitor logs: aws logs tail {CLOUDWATCH_LOG_GROUP_DRIFT} --follow")
        print("  3. Test manually in the next cell")


### 7.3 Verify deployment

In [ ]:
# Load deployment configuration (optional) and resolve account ID
import json
import boto3
from pathlib import Path

config_path = project_root / 'drift_monitoring_config.json'

if config_path.exists():
    with open(config_path) as f:
        config = json.load(f)
    
    # Get actual account ID from STS
    sts = boto3.client('sts', region_name=REGION)
    account_id = sts.get_caller_identity()['Account']
    
    # Replace <ACCOUNT_ID> placeholder in all ARN values
    for key, value in config.items():
        if isinstance(value, str) and '<ACCOUNT_ID>' in value:
            config[key] = value.replace('<ACCOUNT_ID>', account_id)

    print("✅ Drift Monitoring Infrastructure Deployed")
    print("=" * 80)
    print(f"Account ID: {account_id}")
    print(f"SNS Topic ARN: {config['sns_topic_arn']}")
    print(f"Lambda Function ARN: {config['lambda_function_arn']}")
    print(f"EventBridge Rule ARN: {config['eventbridge_rule_arn']}")
    print(f"Schedule: {config.get('schedule', config.get('schedule_expression', 'N/A'))}")
    print(f"Data Drift Threshold: {float(config['data_drift_threshold']) * 100}%")
    print(f"Model Drift Threshold: {float(config['model_drift_threshold']) * 100}% degradation")
    
    if config.get('email'):
        print(f"\n⚠️ IMPORTANT: Check your email ({config['email']}) and confirm SNS subscription!")
else:
    print("ℹ️  Config file not found (drift_monitoring_config.json)")
    print("   This is OK if you're testing manually or deployed via different method")
    print(f"   Expected location: {config_path}")


### 7.4 Manual trigger (test)

In [ ]:
import boto3
import json

lambda_client = boto3.client('lambda', region_name=REGION)

print("🧪 Testing drift monitoring Lambda function...")
print("=" * 80)

try:
    response = lambda_client.invoke(
        FunctionName=DRIFT_LAMBDA_NAME,
        InvocationType='RequestResponse',
        LogType='Tail'
    )

    payload = json.loads(response['Payload'].read())
    
    if payload.get('statusCode') == 200:
        body = json.loads(payload['body'])
        
        print("✅ Drift monitoring check completed successfully")
        print("")
        print(f"Timestamp: {body['timestamp']}")
        print("")
        
        # Data drift results
        if body.get('data_drift'):
            dd = body['data_drift']
            print("📊 DATA DRIFT RESULTS:")
            print(f"  Features Analyzed: {dd.get('features_analyzed', 'N/A')}")
            print(f"  Drifted Features: {dd.get('drifted_features_count', 'N/A')}")
            print(f"  Drift Percentage: {dd.get('drift_percentage', 0):.1f}%")
            print(f"  Average PSI: {dd.get('avg_psi', 0):.4f}")
            print(f"  Max PSI: {dd.get('max_psi', 0):.4f}")
            print(f"  Status: {'🚨 DRIFT DETECTED' if dd.get('detected') else '✓ No drift'}")
            
            if dd.get('drifted_features'):
                print("\n  Top Drifted Features:")
                for feat in dd['drifted_features']:
                    print(f"    - {feat['feature']}: drift_score={feat['drift_score']:.4f}")
        else:
            print("⚠️ Data drift: Insufficient samples")
        
        print("")
        
        # Model drift results
        if body.get('model_drift'):
            md = body['model_drift']
            print("🎯 MODEL DRIFT RESULTS:")
            print(f"  Baseline ROC-AUC: {md.get('baseline_roc_auc', 0):.4f}")
            print(f"  Current ROC-AUC: {md.get('current_roc_auc', 0):.4f}")
            print(f"  Degradation: {md.get('degradation', 0):.4f} ({md.get('degradation_pct', 0):.1f}%)")
            print(f"  Status: {'🚨 DRIFT DETECTED' if md.get('detected') else '✓ No drift'}")
        else:
            print("⚠️ Model drift: Insufficient ground truth samples")
        
        print("")
        print(f"Alert Sent: {'Yes' if body.get('alert_sent') else 'No'}")
        
    else:
        print(f"❌ Lambda returned error: {payload}")

except Exception as e:
    print(f"❌ Error invoking Lambda: {e}")
    import traceback
    traceback.print_exc()

### 7.5 View CloudWatch logs

In [ ]:
import boto3
from datetime import datetime

logs_client = boto3.client('logs', region_name=REGION)
LOG_GROUP = CLOUDWATCH_LOG_GROUP_DRIFT

print('Recent Lambda execution logs:')
print('=' * 80)

try:
    streams = logs_client.describe_log_streams(
        logGroupName=LOG_GROUP,
        orderBy='LastEventTime',
        descending=True,
        limit=1
    )

    if streams['logStreams']:
        stream_name = streams['logStreams'][0]['logStreamName']
        print(f'Latest log stream: {stream_name}')
        print()

        events = logs_client.get_log_events(
            logGroupName=LOG_GROUP,
            logStreamName=stream_name,
            limit=50
        )

        for event in events['events']:
            ts = datetime.fromtimestamp(event['timestamp'] / 1000).strftime('%H:%M:%S')
            msg = event['message'].rstrip()
            print(f'[{ts}] {msg}')
    else:
        print('No logs found yet. Run the test cell above to generate logs.')

except logs_client.exceptions.ResourceNotFoundException:
    print(f'Log group {LOG_GROUP} not found. Deploy and test the Lambda first.')
except Exception as e:
    print(f'Error fetching logs: {e}')

### 7.6 Update thresholds without redeploying

In [ ]:
import boto3

lambda_client = boto3.client('lambda', region_name=REGION)

# Get AWS Account ID
sts = boto3.client('sts', region_name=REGION)
account_id = sts.get_caller_identity()['Account']

# Build SNS Topic ARN dynamically
sns_topic_arn = f"arn:aws:sns:{REGION}:{account_id}:fraud-detection-drift-alerts"

# New configuration (adjust as needed)
NEW_DATA_DRIFT_THRESHOLD = 0.15  # Lower threshold = more sensitive
NEW_MODEL_DRIFT_THRESHOLD = 0.03  # 3% degradation

print("🔧 Updating Lambda configuration...")
print(f"Account ID: {account_id}")
print(f"SNS Topic ARN: {sns_topic_arn}")

try:
    response = lambda_client.update_function_configuration(
        FunctionName=DRIFT_LAMBDA_NAME,
        Environment={
            'Variables': {
                'ATHENA_DATABASE': ATHENA_DATABASE,
                'ATHENA_OUTPUT_S3': ATHENA_OUTPUT_S3,
                'SNS_TOPIC_ARN': sns_topic_arn,
                'DATA_DRIFT_THRESHOLD': str(NEW_DATA_DRIFT_THRESHOLD),
                'MODEL_DRIFT_THRESHOLD': str(NEW_MODEL_DRIFT_THRESHOLD),
                'BASELINE_ROC_AUC': '0.92'
            }
        }
    )
    
    print("✅ Configuration updated successfully")
    print(f"  Data drift threshold: PSI >= {NEW_DATA_DRIFT_THRESHOLD}")
    print(f"  Model drift threshold: {NEW_MODEL_DRIFT_THRESHOLD * 100}% degradation")
    
except Exception as e:
    print(f"❌ Failed to update configuration: {e}")
    print(f"\nTroubleshooting:")
    print(f"  1. Ensure Lambda function exists: {DRIFT_LAMBDA_NAME}")
    print(f"  2. Check you have IAM permissions to update Lambda config")
    print(f"  3. Verify SNS topic exists: {sns_topic_arn}")


### 7.7 Disable / re-enable schedule

In [ ]:
import boto3

events = boto3.client('events', region_name=REGION)

# Set to 'ENABLED' or 'DISABLED'
DESIRED_STATE = 'ENABLED'  # Change to 'DISABLED' to stop monitoring

try:
    if DESIRED_STATE == 'DISABLED':
        events.disable_rule(Name=EVENTBRIDGE_RULE_NAME)
        print("⏸️ Drift monitoring DISABLED")
    else:
        events.enable_rule(Name=EVENTBRIDGE_RULE_NAME)
        print("▶️ Drift monitoring ENABLED")
        
    # Show current status
    rule = events.describe_rule(Name=EVENTBRIDGE_RULE_NAME)
    print(f"\nCurrent status: {rule['State']}")
    print(f"Schedule: {rule['ScheduleExpression']}")
    
except Exception as e:
    print(f"❌ Error: {e}")

### 7.8 Tear down infrastructure

In [ ]:
# Delete all drift monitoring infrastructure
import subprocess
import os

print("This will delete ALL drift monitoring resources.")
print("Make sure you want to do this!")
print("")

DELETE_ECR = "no"  # Set to "yes" to also delete ECR images

result = subprocess.run(
    ['bash', 'scripts/delete_infrastructure.sh', DELETE_ECR],
    cwd=project_root,
    env={**os.environ, 'AWS_REGION': REGION},
    capture_output=False,
    text=True,
)

if result.returncode == 0:
    print("\n✅ Infrastructure deleted successfully")
else:
    print("\n❌ Deletion failed or cancelled")


---